## **Tareas a resolver:**

**Clasificación binaria: mentira o verdad**

**Predicción del hablante: de qué país es el mensaje**

### **Transformers**


##### **Tarea 1: predicción veracidad**


In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -------------------------------------
# Configuración Global
# -------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 10         # Aumentado a 10 como pediste
PATIENCE = 3        # Early Stopping: Si no mejora en 3 épocas, paramos
LR = 2e-5
MAX_LEN = 128
DATA_PATH = "data/train_preprocessed.parquet"

models_to_compare = [
    "distilbert-base-uncased",
    "distilroberta-base"
]

print(f"Usando dispositivo: {DEVICE}")

# -------------------------------------
# 1. Carga y Balanceo de Datos (Oversampling)
# -------------------------------------
df = pd.read_parquet(DATA_PATH)
df = df[df["sender_labels"].astype(str).str.lower().isin(["true", "false"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0}).values

le = LabelEncoder()
df['target'] = le.fit_transform(y_raw)

# Split
X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    df["messages"], df["target"], stratify=df["target"], test_size=0.2, random_state=42
)

# Oversampling en Train
train_df = pd.DataFrame({'text': X_train_raw, 'label': y_train_raw})
df_false = train_df[train_df['label'] == 0]
df_true = train_df[train_df['label'] == 1]

# Igualamos False a True
df_false_over = df_false.sample(len(df_true), replace=True, random_state=42)
df_balanced = pd.concat([df_true, df_false_over], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

X_train = df_balanced['text'].values
y_train = df_balanced['label'].values
X_val = X_val_raw.values
y_val = y_val.values

print(f"Datos preparados. Train (Balanceado): {len(X_train)} | Val: {len(X_val)}")

# -------------------------------------
# 2. Clases Utilitarias
# -------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_model(model_name, X_train, y_train, X_val, y_val):
    print(f"\n{'='*40}")
    print(f"PROCESANDO: {model_name}")
    print(f"{'='*40}")
    
    # Cargar Tokenizer y Modelo específicos
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(DEVICE)
    
    # Dataloaders
    train_ds = TransformerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = TransformerDataset(X_val, y_val, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    
    # Variables de control
    best_mcc = -1
    best_epoch = 0
    patience_counter = 0
    history = []
    
    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            model.zero_grad()
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- Validation ---
        model.eval()
        y_true, y_pred = [], []
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                y_pred.extend(preds)
                y_true.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        mcc = matthews_corrcoef(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")
        acc = accuracy_score(y_true, y_pred)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Val Loss: {avg_val_loss:.4f} | MCC: {mcc:.4f} | F1: {f1:.4f}")
        
        history.append({
            "Model": model_name,
            "Epoch": epoch + 1,
            "Val Loss": avg_val_loss,
            "MCC": mcc,
            "F1": f1,
            "Accuracy": acc
        })
        
        # --- Early Stopping & Checkpoint ---
        if mcc > best_mcc:
            best_mcc = mcc
            best_epoch = epoch + 1
            patience_counter = 0
            # Aquí podrías guardar el modelo: torch.save(model.state_dict(), f"{model_name}_best.pt")
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print(f"Early Stopping activado. No mejora desde Epoch {best_epoch}.")
            break
            
    print(f"Mejor MCC para {model_name}: {best_mcc:.4f} (Epoch {best_epoch})")
    return history

# -------------------------------------
# 3. Ejecución y Comparativa
# -------------------------------------
all_results = []

for m in models_to_compare:
    res = train_model(m, X_train, y_train, X_val, y_val)
    all_results.extend(res)

# Crear DataFrame final
df_res = pd.DataFrame(all_results)
print("\n" + "="*50)
print("TABLA COMPARATIVA FINAL")
print("="*50)

# Mostrar la mejor fila de cada modelo (basado en MCC)
best_rows = df_res.loc[df_res.groupby("Model")["MCC"].idxmax()].sort_values("MCC", ascending=False)
print(best_rows[["Model", "Epoch", "Accuracy", "F1", "MCC"]])

Usando dispositivo: cuda
Datos preparados. Train (Balanceado): 18194 | Val: 2379

PROCESANDO: distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10 | Val Loss: 0.4026 | MCC: 0.0573 | F1: 0.5276
Epoch 2/10 | Val Loss: 0.4854 | MCC: 0.0628 | F1: 0.5308
Epoch 3/10 | Val Loss: 0.5940 | MCC: 0.0668 | F1: 0.5329
Epoch 4/10 | Val Loss: 0.5596 | MCC: 0.0253 | F1: 0.5124
Epoch 5/10 | Val Loss: 0.5690 | MCC: 0.0790 | F1: 0.5395
Epoch 6/10 | Val Loss: 0.4614 | MCC: 0.0382 | F1: 0.5178
Epoch 7/10 | Val Loss: 0.5046 | MCC: 0.0424 | F1: 0.5194
Epoch 8/10 | Val Loss: 0.4310 | MCC: 0.0609 | F1: 0.5224
Early Stopping activado. No mejora desde Epoch 5.
Mejor MCC para distilbert-base-uncased: 0.0790 (Epoch 5)

PROCESANDO: distilroberta-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10 | Val Loss: 0.6459 | MCC: 0.0664 | F1: 0.5031
Epoch 2/10 | Val Loss: 0.5894 | MCC: 0.0808 | F1: 0.5396
Epoch 3/10 | Val Loss: 0.7481 | MCC: 0.1190 | F1: 0.5514
Epoch 4/10 | Val Loss: 0.6199 | MCC: 0.0750 | F1: 0.5373
Epoch 5/10 | Val Loss: 0.5200 | MCC: 0.0719 | F1: 0.5359
Epoch 6/10 | Val Loss: 0.5618 | MCC: 0.0670 | F1: 0.5332
Early Stopping activado. No mejora desde Epoch 3.
Mejor MCC para distilroberta-base: 0.1190 (Epoch 3)

TABLA COMPARATIVA FINAL
                      Model  Epoch  Accuracy        F1       MCC
10       distilroberta-base      3  0.887768  0.551430  0.119035
4   distilbert-base-uncased      5  0.924758  0.539465  0.078980


DistilRoBERTa presentó el desempeño más robusto (MCC 0.1190 y F1 0.5514). Aunque DistilBERT alcanzó una exactitud del 92.47%, su bajo MCC (0.0790) indica un claro sesgo hacia la clase mayoritaria (True), fallando en la identificación de mentiras. Por esto, el coeficiente de Matthews es la métrica óptima para esta tarea. El oversampling resulta insuficiente para modelar la complejidad del engaño sin técnicas complementarias.

Para intentar reducir el sesgo, reemplazamos el oversampling por Class Weights, penalizando severamente los errores en la clase minoritaria dentro de la loss function. También ajustamos el umbral de decisión para decidir si es una mentira (más de 0.5) para maximizar directamente el MCC.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight # <--- CAMBIO: Necesario para calcular pesos
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# -------------------------------------
# Configuración Global
# -------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 10
PATIENCE = 3
LR = 2e-5
MAX_LEN = 128
DATA_PATH = "data/train_preprocessed.parquet"

models_to_compare = ["distilroberta-base"] # Probamos con el mejor de la ronda anterior

print(f"Usando dispositivo: {DEVICE}")

# -------------------------------------
# 1. Carga de Datos (SIN OVERSAMPLING)
# -------------------------------------
df = pd.read_parquet(DATA_PATH)
df = df[df["sender_labels"].astype(str).str.lower().isin(["true", "false"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0}).values

le = LabelEncoder()
df['target'] = le.fit_transform(y_raw)

# Split normal (Stratified)
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    df["messages"], df["target"], stratify=df["target"], test_size=0.2, random_state=42
)

# <--- CAMBIO IMPORTANTE: Usamos los datos RAW, no los balanceados
X_train = X_train_raw.values
y_train = y_train_raw.values
X_val = X_val_raw.values
y_val = y_val_raw.values

print(f"Datos preparados (Originales). Train: {len(X_train)} | Val: {len(X_val)}")

# -------------------------------------
# 2. Calcular Pesos de Clase (Propuesta 1)
# -------------------------------------
# Calculamos el peso inverso para compensar el desbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
# Convertimos a tensor float y movemos al dispositivo
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print(f"Pesos calculados: Clase 0 (False): {class_weights[0]:.2f} | Clase 1 (True): {class_weights[1]:.2f}")
# Probablemente verás algo como: Clase 0: ~10.0 | Clase 1: ~0.5

# -------------------------------------
# 3. Clases Utilitarias (Dataset igual que antes)
# -------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# -------------------------------------
# 4. Función Auxiliar para Threshold Moving (Propuesta 2)
# -------------------------------------
def find_best_threshold(y_true, probs_class_0):
    """
    Busca el umbral óptimo para clasificar la clase minoritaria (0).
    Si prob_clase_0 > threshold -> Predice 0 (False)
    """
    best_thresh = 0.5
    best_mcc = -1
    
    thresholds = np.arange(0.1, 0.9, 0.05) # Probamos de 0.1 a 0.9
    
    for thresh in thresholds:
        # Si la probabilidad de ser 0 es mayor al umbral, es 0. Si no, es 1.
        preds = np.where(probs_class_0 > thresh, 0, 1)
        mcc = matthews_corrcoef(y_true, preds)
        
        if mcc > best_mcc:
            best_mcc = mcc
            best_thresh = thresh
            
    return best_thresh, best_mcc

# -------------------------------------
# 5. Loop de Entrenamiento Modificado
# -------------------------------------
def train_model_weighted(model_name, X_train, y_train, X_val, y_val, class_weights_tensor):
    print(f"\n{'='*40}")
    print(f"ENTRENANDO CON CLASS WEIGHTS: {model_name}")
    print(f"{'='*40}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(DEVICE)
    
    train_ds = TransformerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = TransformerDataset(X_val, y_val, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    optimizer = AdamW(model.parameters(), lr=LR)
    
    # <--- CAMBIO: Aplicamos los pesos a la función de pérdida
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    best_mcc_val = -1
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            model.zero_grad()
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- Validation (con Probabilidades) ---
        model.eval()
        y_true_list = []
        probs_class_0_list = [] # Guardamos probabilidad de ser "False" (Clase 0)
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                
                # <--- CAMBIO: Obtenemos probabilidades con Softmax
                probs = F.softmax(outputs.logits, dim=1)
                
                # Guardamos probabilidad de la clase 0 (la columna 0)
                probs_class_0_list.extend(probs[:, 0].cpu().numpy())
                y_true_list.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        
        # Conversión a arrays numpy
        y_true_np = np.array(y_true_list)
        probs_0_np = np.array(probs_class_0_list)
        
        # <--- CAMBIO: Buscamos el mejor umbral dinámicamente
        best_thresh, current_mcc = find_best_threshold(y_true_np, probs_0_np)
        
        # Generamos predicciones finales con ese umbral para las otras métricas
        final_preds = np.where(probs_0_np > best_thresh, 0, 1)
        f1 = f1_score(y_true_np, final_preds, average="macro")
        acc = accuracy_score(y_true_np, final_preds)
        
        print(f"Epoch {epoch+1} | Loss: {avg_val_loss:.4f} | Best MCC: {current_mcc:.4f} (Thresh: {best_thresh:.2f}) | F1: {f1:.4f}")
        
        # Early Stopping basado en MCC
        if current_mcc > best_mcc_val:
            best_mcc_val = current_mcc
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print("Early Stopping activado.")
            break

    print(f"Mejor MCC alcanzado: {best_mcc_val:.4f}")

# -------------------------------------
# 6. Ejecución
# -------------------------------------
for m in models_to_compare:
    train_model_weighted(m, X_train, y_train, X_val, y_val, weights_tensor)

Usando dispositivo: cuda
Datos preparados (Originales). Train: 9515 | Val: 2379
Pesos calculados: Clase 0 (False): 11.38 | Clase 1 (True): 0.52

ENTRENANDO CON CLASS WEIGHTS: distilroberta-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 1.1443 | Best MCC: 0.0000 (Thresh: 0.10) | F1: 0.4888
Epoch 2 | Loss: 0.8907 | Best MCC: 0.0837 (Thresh: 0.10) | F1: 0.5303
Epoch 3 | Loss: 0.7070 | Best MCC: 0.1083 (Thresh: 0.40) | F1: 0.5076
Epoch 4 | Loss: 0.9068 | Best MCC: 0.1232 (Thresh: 0.25) | F1: 0.5537
Epoch 5 | Loss: 1.1176 | Best MCC: 0.1130 (Thresh: 0.10) | F1: 0.5546
Epoch 6 | Loss: 1.2189 | Best MCC: 0.1113 (Thresh: 0.60) | F1: 0.5521
Epoch 7 | Loss: 1.7343 | Best MCC: 0.1038 (Thresh: 0.10) | F1: 0.5506
Early Stopping activado.
Mejor MCC alcanzado: 0.1232


Con esta prueba logramos subir el MCC a 0.1232, que es nuestro mejor resultado hasta ahora, pero la mejora ha sido muy pequeña. A partir del tercer epoch el loss se dispara, lo que significa que está empezando a memorizar en lugar de aprender.

El modelo intenta predecir si una frase es verdad o mentira, pero sin mas cotexto la misma frase puede ser verdad o mentira. Por eso a continuación vamos a añadir los sender y receiver labels. En el preprocesado añadimos una columna con el contexto que contiene datos con este formato: 

Emisor -> Receptor: Mensaje

In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# -------------------------------------
# Configuración Global
# -------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 10
PATIENCE = 3
LR = 2e-5
MAX_LEN = 128
DATA_PATH = "data/train_with_context.parquet" 

models_to_compare = ["distilroberta-base"]

print(f"Usando dispositivo: {DEVICE}")

# -------------------------------------
# 1. Carga de Datos (Con Contexto)
# -------------------------------------
df = pd.read_parquet(DATA_PATH)
df = df[df["sender_labels"].astype(str).str.lower().isin(["true", "false"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0}).values

le = LabelEncoder()
df['target'] = le.fit_transform(y_raw)

# <--- CAMBIO 2: Usamos la columna 'text_context' en lugar de 'messages'
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    df["text_context"], df["target"], stratify=df["target"], test_size=0.2, random_state=42
)

X_train = X_train_raw.values
y_train = y_train_raw.values
X_val = X_val_raw.values
y_val = y_val_raw.values

print(f"Datos preparados (Contexto Inyectado). Train: {len(X_train)} | Val: {len(X_val)}")
# Muestra un ejemplo para verificar
print(f"Ejemplo de entrada: {X_train[0]}")

# -------------------------------------
# 2. Calcular Pesos de Clase
# -------------------------------------
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Pesos: Clase 0 (False): {class_weights[0]:.2f} | Clase 1 (True): {class_weights[1]:.2f}")

# -------------------------------------
# 3. Dataset y Funciones
# -------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def find_best_threshold(y_true, probs_class_0):
    best_thresh = 0.5
    best_mcc = -1
    thresholds = np.arange(0.1, 0.9, 0.05)
    
    for thresh in thresholds:
        preds = np.where(probs_class_0 > thresh, 0, 1)
        mcc = matthews_corrcoef(y_true, preds)
        if mcc > best_mcc:
            best_mcc = mcc
            best_thresh = thresh
    return best_thresh, best_mcc

# -------------------------------------
# 4. Loop de Entrenamiento
# -------------------------------------
def train_model_weighted(model_name, X_train, y_train, X_val, y_val, class_weights_tensor):
    print(f"\n{'='*40}")
    print(f"ENTRENANDO (Context + Weights): {model_name}")
    print(f"{'='*40}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(DEVICE)
    
    train_ds = TransformerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = TransformerDataset(X_val, y_val, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    best_mcc_val = -1
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            model.zero_grad()
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- Validation ---
        model.eval()
        y_true_list = []
        probs_class_0_list = []
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                
                probs = F.softmax(outputs.logits, dim=1)
                probs_class_0_list.extend(probs[:, 0].cpu().numpy())
                y_true_list.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        y_true_np = np.array(y_true_list)
        probs_0_np = np.array(probs_class_0_list)
        
        best_thresh, current_mcc = find_best_threshold(y_true_np, probs_0_np)
        
        # Métricas con el mejor threshold
        final_preds = np.where(probs_0_np > best_thresh, 0, 1)
        f1 = f1_score(y_true_np, final_preds, average="macro")
        
        print(f"Epoch {epoch+1} | Loss: {avg_val_loss:.4f} | Best MCC: {current_mcc:.4f} (Thresh: {best_thresh:.2f}) | F1: {f1:.4f}")
        
        if current_mcc > best_mcc_val:
            best_mcc_val = current_mcc
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print("Early Stopping activado.")
            break

    print(f"Mejor MCC Final: {best_mcc_val:.4f}")

# -------------------------------------
# 5. Ejecución
# -------------------------------------
for m in models_to_compare:
    train_model_weighted(m, X_train, y_train, X_val, y_val, weights_tensor)


ENTRENANDO (Stabilized): distilroberta-base | LR: 5e-06


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 1.2037 | Best MCC: 0.0000 (Thresh: 0.15) | F1: 0.4888
Epoch 2 | Loss: 1.0305 | Best MCC: 0.1844 (Thresh: 0.15) | F1: 0.5908
Epoch 3 | Loss: 0.8772 | Best MCC: 0.1892 (Thresh: 0.25) | F1: 0.5913
Epoch 4 | Loss: 0.9336 | Best MCC: 0.1759 (Thresh: 0.10) | F1: 0.5850
Epoch 5 | Loss: 1.1102 | Best MCC: 0.1550 (Thresh: 0.10) | F1: 0.5705
Epoch 6 | Loss: 1.2296 | Best MCC: 0.1866 (Thresh: 0.15) | F1: 0.5932
Epoch 7 | Loss: 1.3164 | Best MCC: 0.1617 (Thresh: 0.45) | F1: 0.5725
Early Stopping activado.
Mejor MCC Final: 0.1892


La inclusión del contexto (Emisor -> Receptor) es clave para elevar el MCC de 0.12 a 0.189, demostrando que el engaño depende de la interacción y no solo del texto. El uso de pesos de clase y el ajuste del umbral ayudan al modelo identificar mentiras en un entorno de desbalance (1:21), aunque el rápido aumento del error de validación tras el tercer epoch indica que el modelo empieza a memorizar. Estos resultados confirman al MCC como la métrica más fiable.

##### **Tarea 2: predicción del hablante**


Ahora vamos a abordar la tarea de predicción del hablante utilizando modelos basados en Transformers, concretamente DistilBERT y DistilRoBERTa (adaptados para entrenar en CPU).

In [1]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install hf_xet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# CONFIGURACIÓN (optimizado para CPU).
# Forzamos el uso de CPU para evitar problemas de compatibilidad.
DEVICE = "cpu"

# Limitamos el número de hilos para evitar sobrecargar la CPU.
torch.set_num_threads(4)

BATCH_SIZE = 8       # Tamaño de batch pequeño para reducir el consumo de memoria.
EPOCHS = 6           # Número de epochs reducido ya que los transformers suelen converger rápido.
PATIENCE = 2         # Early stopping, número de epochs sin mejora antes de parar.
LR = 2e-5            # Learning rate típico para fine-tuning de transformers.
MAX_LEN = 128        # Longitud máxima de los mensajes (suficiente para textos cortos).

# Ruta al dataset preprocesado.
DATA_PATH = "data/train_preprocessed.parquet"

# Estos son los modelos transformer que vamos a comparar.
models_to_compare = [
    "distilbert-base-uncased",
    "distilroberta-base"
]

print(f"Usando dispositivo: {DEVICE}")

# 1. CARGA Y PREPROCESADO DEL DATASET
# Primero cargamos el dataset desde un archivo Parquet.
df = pd.read_parquet(DATA_PATH)

# La columna "speakers" aparece en distintos formatos.
# Esta función normaliza el valor para obtener un único speaker por mensaje.
def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    if isinstance(x, str) and x.startswith("["):
        return x.strip("[]").replace("'", "").split(",")[0].strip()
    return str(x)

# Ahora aplicamos la normalización a toda la columna.
df["speakers"] = df["speakers"].apply(flatten_speaker)

# Después codificamos los hablantes como etiquetas numéricas.
le = LabelEncoder()
y = le.fit_transform(df["speakers"])
num_classes = len(le.classes_)

# Textos de entrada.
X = df["messages"].astype(str).values

# División train / validation con estratificación:
# Esto garantiza que la proporción de hablantes se mantenga.
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Calculamos pesos de clase para disminuir el desbalance entre hablantes.
class_weights = compute_class_weight("balanced", classes=np.unique(y), y=y)

# Convertimos los pesos a tensor para usarlos en PyTorch.
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)


# 2. DEFINICIÓN DEL DATASET PERSONALIZADO
# Esta clase se encarga de tokenizar los textos y devolver los tensores que necesita el modelo transformer.
class SpeakerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        t = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenización del texto:
        enc = self.tokenizer.encode_plus(
            t, add_special_tokens=True, max_length=self.max_len,
            padding="max_length", truncation=True,
            return_attention_mask=True, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }


# 3. FUNCIÓN DE ENTRENAMIENTO
# Esta función entrena y evalúa un modelo transformer concreto.
def train_model(model_name):
    print("\n" + "="*50)
    print(f" ENTRENANDO {model_name} ")
    print("="*50)
    # Cargamos el tokenizador correspondiente al modelo.
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Cargamos el modelo preentrenado con una cabeza de clasificación.
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes
    )
    model.to(DEVICE)

    # Creamos los datasets y dataloaders.
    train_ds = SpeakerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = SpeakerDataset(X_val, y_val, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    # Optimizador AdamW, estándar para transformers.
    optimizer = AdamW(model.parameters(), lr=LR)

    # Función de pérdida con pesos de clase.
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    best_f1 = -1
    patience_counter = 0
    results = []

    # Bucle de entrenamiento.
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # FASE DE VALIDACIÓN
        model.eval()
        y_true, y_pred = [], []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                logits = model(input_ids, attention_mask=mask).logits
                preds = torch.argmax(logits, dim=1).cpu().numpy()

                y_pred.extend(preds)
                y_true.extend(labels.cpu().numpy())

        # Métricas de evaluación.
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")

        print(f"Epoch {epoch+1}/{EPOCHS} | Acc={acc:.4f} | F1_macro={f1:.4f}")

        results.append({
            "Model": model_name,
            "Epoch": epoch+1,
            "Accuracy": acc,
            "F1_macro": f1
        })

    
        # EARLY STOPPING
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping activado.")
            break

    print(f"Mejor F1 para {model_name}: {best_f1:.4f}")
    return results


# 4. EJECUCIÓN DE LOS EXPERIMENTOS
all_results = []

# Entrenamos y evaluamos cada modelo
for model_name in models_to_compare:
    r = train_model(model_name)
    all_results.extend(r)

# Finalmente, creamos una tabla con los mejores resultados por modelo
df_res = pd.DataFrame(all_results)
best = df_res.loc[df_res.groupby("Model")["F1_macro"].idxmax()]
print("\nRESULTADOS FINALES\n")
print(best)

Usando dispositivo: cpu

 ENTRENANDO distilbert-base-uncased 


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

C:\Users\Oihane\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Oihane\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of DistilB

Epoch 1/6 | Acc=0.2984 | F1_macro=0.2647
Epoch 2/6 | Acc=0.3615 | F1_macro=0.3172
Epoch 3/6 | Acc=0.3598 | F1_macro=0.3172
Epoch 4/6 | Acc=0.3615 | F1_macro=0.3235
Epoch 5/6 | Acc=0.3602 | F1_macro=0.3221
Epoch 6/6 | Acc=0.3472 | F1_macro=0.3053
Early stopping activado.
Mejor F1 para distilbert-base-uncased: 0.3235

 ENTRENANDO distilroberta-base 


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/6 | Acc=0.2762 | F1_macro=0.2580
Epoch 2/6 | Acc=0.3127 | F1_macro=0.2809
Epoch 3/6 | Acc=0.3577 | F1_macro=0.3184
Epoch 4/6 | Acc=0.3695 | F1_macro=0.3288
Epoch 5/6 | Acc=0.3380 | F1_macro=0.3179
Epoch 6/6 | Acc=0.3653 | F1_macro=0.3159
Early stopping activado.
Mejor F1 para distilroberta-base: 0.3288

RESULTADOS FINALES

                     Model  Epoch  Accuracy  F1_macro
3  distilbert-base-uncased      4  0.361496  0.323509
9       distilroberta-base      4  0.369483  0.328763


Ambos modelos superan de forma clara los resultados obtenidos en la primera iteración con técnicas de Deep Learning clásicas (CNN y LSTM con Word2Vec). En concreto, DistilBERT alcanza un F1-macro de 0.324, mientras que DistilRoBERTa obtiene el mejor rendimiento global con un F1-macro de 0.329 y una accuracy cercana al 37%. Con esto podemos ver que los Transformers tienen la capacidad para capturar información semántica más rica incluso en textos cortos y homogéneos como los que tenemos en Diplomacy. Sin embargo, el incremento de rendimiento viene acompañado de un coste computacional significativamente mayor, especialmente en entornos sin GPU.

Hemos decidido repetir el experimento pero esta vez adaptando el entrenamiento para ejecutarse en GPU. El objetivo de este paso es, por un lado, reducir el tiempo de entrenamiento y, por otro, comprobar si un entrenamiento más eficiente permite alcanzar mejores resultados mediante un mayor número de epoch y una exploración más estable del espacio de parámetros. Para ello, mantenemos la misma arquitectura (DistilBERT y DistilRoBERTa), pero incorporamos técnicas específicas para GPU como el uso de mixed precision y un tamaño de batch mayor.

In [ ]:
# CÓDIGO DE ENTRENAMIENTO CON GPU
# Aquí vamos a comparar los modelos DistilBERT y DistilRoBERTa con GPU
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm


# CONFIGURACIÓN (GPU)
# Usamos GPU si está disponible, en caso contrario se usa CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {DEVICE}")

BATCH_SIZE = 16          # Tamaño de batch mayor aprovechando la capacidad de la GPU.
EPOCHS = 8               # Número de epochs algo mayor que en CPU.
PATIENCE = 2             # Early stopping, número de epochs sin mejora en MCC.
LR = 2e-5                # Learning rate típico para transformers.
MAX_LEN = 128            # Longitud máxima de los textos.

DATA_PATH = "data/train_preprocessed.parquet"

# Los modelos a comparar en esta iteración.
models_to_compare = [
    "distilbert-base-uncased",
    "distilroberta-base"
]


# 1. CARGA Y PREPROCESADO DEL DATASET
# En este paso cargamos el dataset y preparamos las etiquetas como hemos hecho antes.
df = pd.read_parquet(DATA_PATH)

def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    if isinstance(x, str) and x.startswith("["):
        return x.strip("[]").replace("'", "").split(",")[0].strip()
    return str(x)

df["speakers"] = df["speakers"].apply(flatten_speaker)

# Labels
le = LabelEncoder()
y = le.fit_transform(df["speakers"])
num_classes = len(le.classes_)

X = df["messages"].astype(str).values

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y),
    y=y
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)


# 2. DEFINICIÓN DEL DATASET PERSONALIZADO
# Esta clase se encarga de tokenizar los textos y preparar los tensores..
class SpeakerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Tokenizamos el texto correspondiente al índice idx.
        enc = self.tokenizer(
            str(self.texts[idx]),
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


# 3. FUNCIÓN DE ENTRENAMIENTO Y EVALUACIÓN
# En esta función entrenamos y evaluamos un modelo transformer específico (parecido a lo que hemos hecho antes).
def train_model(model_name):
    print("\n" + "="*50)
    print(f" ENTRENANDO {model_name} ")
    print("="*50)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes
    ).to(DEVICE)

    train_ds = SpeakerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = SpeakerDataset(X_val, y_val, tokenizer, MAX_LEN)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, num_workers=0
    )

    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    # GradScaler para entrenamiento en precisión mixta.
    scaler = torch.cuda.amp.GradScaler()

    best_mcc = -1
    patience_counter = 0

     # Bucle de entrenamiento.
    for epoch in range(EPOCHS):
       
        # ENTRENAMIENTO
        model.train()
        train_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            # Forward pass con mixed precision
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)

            # Backpropagation escalada para estabilidad numérica.
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

        
        # VALIDACIÓN
        model.eval()
        y_true, y_pred = [], []
        val_loss = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)

                val_loss += loss.item()
                preds = torch.argmax(outputs.logits, dim=1)

                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        # Cálculamos las métricas.
        val_loss /= len(val_loader)
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")
        mcc = matthews_corrcoef(y_true, y_pred)

        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"Val Loss: {val_loss:.4f} | "
            f"MCC: {mcc:.4f} | "
            f"F1: {f1:.4f} | "
            f"Acc: {acc:.4f}"
        )

        # EARLY STOPPING BASADO EN MCC
        if mcc > best_mcc:
            best_mcc = mcc
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(
                f"Early Stopping activado. "
                f"No mejora desde Epoch {epoch+1-PATIENCE}."
            )
            break

    print(f"Mejor MCC para {model_name}: {best_mcc:.4f}")


# 4. EJECUCIÓN DE LOS EXPERIMENTOS
# Entrenamos y evaluamos cada modelo.
for model_name in models_to_compare:
    train_model(model_name)


Usando dispositivo: cuda

 ENTRENANDO distilbert-base-uncased 


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Alba\AppData\Local\Temp\ipykernel_34600\373461215.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/8: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 595/595 [00:43<00:00, 13.66it/s]


Epoch 1/8 | Val Loss: 1.7927 | MCC: 0.1840 | F1: 0.2639 | Acc: 0.3232


Epoch 2/8: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 595/595 [00:43<00:00, 13.78it/s]


Epoch 2/8 | Val Loss: 1.7326 | MCC: 0.2291 | F1: 0.3105 | Acc: 0.3472


...

Epoch 6/8 | Val Loss: 2.2000 | MCC: 0.2330 | F1: 0.3188 | Acc: 0.3560


Epoch 7/8: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 595/595 [00:44<00:00, 13.51it/s]


Epoch 7/8 | Val Loss: 2.5331 | MCC: 0.2421 | F1: 0.3222 | Acc: 0.3749


Epoch 8/8: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 595/595 [00:44<00:00, 13.23it/s]


Epoch 8/8 | Val Loss: 2.7193 | MCC: 0.2268 | F1: 0.3114 | Acc: 0.3430
Mejor MCC para distilroberta-base: 0.2421


Los resultados obtenidos con este entrenamiento muestran una mejora consistente respecto a la versión entrenada en CPU. En el caso de DistilBERT, el modelo alcanza un MCC de 0.239 y un F1-macro cercano a 0.322, mientras que DistilRoBERTa obtiene el mejor rendimiento global con un MCC de 0.242 y un F1-macro de 0.322. Estos valores suponen una mejora respecto a los modelos anteriores y confirman que el uso de GPU no solo acelera el entrenamiento, sino que también permite una optimización más estable del modelo. No obstante, se observa que a partir de ciertas epoch el rendimiento comienza a bajar, por lo que usamos early stopping. Esto demuestra la dificultad de la tarea de identificación del hablante en un contexto altamente desbalanceado y semánticamente complejo.

Dado que en los experimentos anteriores los modelos basados en RoBERTa mostraron un rendimiento ligeramente superior a DistilBERT, decidimos dar un paso más y entrenar una versión completa de RoBERTa-base. En este modelo hemos introducido varias mejoras para estabilizar el entrenamiento y favorecer una mejor convergencia: hemos usado un scheduler con warm-up para el learning rate, congelado las primeras capas del encoder y el entrenado en GPU con mixed precision.

In [ ]:
# ENTRENAMIENTO CON TRANSFORMERS AVANZADO (RoBERTa-base).
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from tqdm import tqdm

# CONFIGURACIÓN GENERAL (igual que en modelos GPU anteriores).
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 8
PATIENCE = 2
LR = 2e-5
MAX_LEN = 128

# Ahora usamos un modelo más grande y expresivo
MODEL_NAME = "roberta-base"

print(f"Usando dispositivo: {DEVICE}")


# 1. CARGA Y PREPROCESADO DE DATOS
# (Mismo procedimiento que en los modelos anteriores).
df = pd.read_parquet("data/train_preprocessed.parquet")

def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    if isinstance(x, str) and x.startswith("["):
        return x.strip("[]").replace("'", "").split(",")[0].strip()
    return str(x)

df["speakers"] = df["speakers"].apply(flatten_speaker)

# Codificación de etiquetas (igual que antes).
le = LabelEncoder()
y = le.fit_transform(df["speakers"])
num_classes = len(le.classes_)

X = df["messages"].astype(str).values

# Split estratificado (igual que antes).
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Cálculo de pesos de clase para tratar el desbalance (igual que antes).
cls_weights = compute_class_weight("balanced", classes=np.unique(y), y=y)
cls_weights = torch.tensor(cls_weights, dtype=torch.float32).to(DEVICE)


# 2. DATASET
# (Estructura idéntica a la usada en DistilBERT / DistilRoBERTa).
class SpeakerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self): 
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


# 3. TOKENIZER Y MODELO
# Tokenizador específico de RoBERTa (igual que antes, pero para otro modelo).
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Cargamos RoBERTa-base con cabeza de clasificación.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
).to(DEVICE)

# CONGELADO PARCIAL DEL ENCODER
# En lugar de fine-tuning completo, congelamos las primeras capas del encoder.
# Esto mejora la estabilidad del entrenamiento y reduce el riesgo de overfitting,
# especialmente en datasets pequeños y con fuerte solapamiento léxico.
for name, param in model.named_parameters():
    if "encoder.layer.0" in name or "encoder.layer.1" in name:
        param.requires_grad = False

# 4. DATALOADERS
# (Igual que en los modelos GPU anteriores)
train_ds = SpeakerDataset(X_train, y_train, tokenizer)
val_ds = SpeakerDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)


# 5. OPTIMIZACIÓN
# Optimizador AdamW (igual que antes).
optimizer = AdamW(model.parameters(), lr=LR)

# Función de pérdida con pesos de clase (igual que antes).
loss_fn = nn.CrossEntropyLoss(weight=cls_weights)

# LEARNING RATE SCHEDULER CON WARM-UP
# Se introduce un scheduler lineal con warm-up para evitar inestabilidad
# en las primeras iteraciones del fine-tuning.
# Durante el warm-up, el learning rate aumenta progresivamente.
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Mixed precision (igual que en el modelo GPU anterior).
scaler = torch.cuda.amp.GradScaler()


# 6. ENTRENAMIENTO
best_mcc = -1
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # Forward pass con mixed precision.
        with torch.cuda.amp.autocast():
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)

        # Backpropagation escalada.
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # actualización del scheduler en cada paso
        scheduler.step()

        train_loss += loss.item()

  
    # VALIDACIÓN
    # (Mismo esquema que en los modelos anteriores).
    model.eval()
    y_true, y_pred = [], []
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            y_pred.extend(preds.cpu().numpy())
            y_true.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    mcc = matthews_corrcoef(y_true, y_pred)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Val Loss: {val_loss:.4f} | "
        f"MCC: {mcc:.4f} | "
        f"F1: {f1:.4f} | "
        f"Acc: {acc:.4f}"
    )

    # EARLY STOPPING BASADO EN MCC (igual que antes)
    if mcc > best_mcc:
        best_mcc = mcc
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping activado.")
        break

print(f"\nMejor MCC obtenido: {best_mcc:.4f}")

Usando dispositivo: cuda


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Alba\AppData\Local\Temp\ipykernel_19928\1299145765.py:123: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/8:   0%|                                                                                                      | 0/595 [00:00<?, ?it/s]C:\Users\Alba\AppData\Local\Temp\ipykernel_19928\1299145765.py:142: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/8: 100%|██████████████████████

Epoch 1/8 | Val Loss: 1.9246 | MCC: 0.0765 | F1: 0.1400 | Acc: 0.1761


Epoch 2/8:   0%|                                                                                                      | 0/595 [00:00<?, ?it/s]C:\Users\Alba\AppData\Local\Temp\ipykernel_19928\1299145765.py:142: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/8: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 595/595 [01:03<00:00,  9.39it/s]


Epoch 2/8 | Val Loss: 1.8041 | MCC: 0.1787 | F1: 0.2525 | Acc: 0.2980


...

Epoch 6/8 | Val Loss: 1.8833 | MCC: 0.2261 | F1: 0.3156 | Acc: 0.3371


Epoch 7/8:   0%|                                                                                                      | 0/595 [00:00<?, ?it/s]C:\Users\Alba\AppData\Local\Temp\ipykernel_19928\1299145765.py:142: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 7/8: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 595/595 [01:07<00:00,  8.78it/s]


Epoch 7/8 | Val Loss: 1.9662 | MCC: 0.2362 | F1: 0.3269 | Acc: 0.3539


Epoch 8/8:   0%|                                                                                                      | 0/595 [00:00<?, ?it/s]C:\Users\Alba\AppData\Local\Temp\ipykernel_19928\1299145765.py:142: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 8/8: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 595/595 [01:08<00:00,  8.67it/s]


Epoch 8/8 | Val Loss: 2.0047 | MCC: 0.2422 | F1: 0.3316 | Acc: 0.3623

Mejor MCC obtenido: 0.2422


Los resultados obtenidos con RoBERTa-base confirman una mejora progresiva respecto a los modelos anteriores. El modelo alcanza un F1-macro de 0.332, una accuracy del 36.2% y un MCC máximo de 0.242, situándose como el mejor modelo hasta el momento. La evolución por epoch muestra una convergencia más estable y un incremento gradual del rendimiento, lo que sugiere que la combinación de un scheduler con warm-up y la congelación parcial de capas ayuda a evitar actualizaciones demasiado agresivas en las primeras fases del entrenamiento. A pesar de que la mejora respecto a DistilRoBERTa es moderada, estos resultados indican que un modelo más profundo y una optimización más controlada nos dejan capturar mejor las diferencias sutiles entre hablantes en un escenario muy desbalanceado.

Después de ver que RoBERTa-base ofrecía los mejores resultados, en este último experimento vamos a centramos en ajustar el entrenamiento con el objetivo de exprimir al máximo su capacidad. Para ello, vamos a hacer varias modificaciones para mejorar la estabilidad y la generalización del modelo: reducción del learning rate, aumento moderado de la longitud máxima de los textos, uso de gradient accumulation para simular un batch efectivo mayor, congelación más agresiva de las primeras capas del modelo y la incorporación de label smoothing para mitigar el efecto del desbalanceo entre clases.

In [ ]:
# MODELO TRANSFORMER AVANZADO CON OPTIMIZACIONES
# RoBERTa-base + Gradient Accumulation + Label Smoothing
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm


# CONFIGURACIÓN GENERAL
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {DEVICE}")

# Modelo base más potente (igual que el anterior, pero con más ajustes).
MODEL_NAME = "roberta-base"    
BATCH_SIZE = 8                 # compensado con accumulation
ACCUM_STEPS = 2                # batch efectivo = 16
EPOCHS = 10
PATIENCE = 2
LR = 1e-5                      # Learning rate más bajo porque RoBERTa es sensible a LR altos.
MAX_LEN = 160                  # secuencias algo más largas para capturar más contexto.
WARMUP_RATIO = 0.1             # Proporción de warm-up.

DATA_PATH = "data/train_preprocessed.parquet"


# 1. CARGA Y PREPROCESADO DEL DATASET
# (Igual que en los modelos anteriores).
df = pd.read_parquet(DATA_PATH)

def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    if isinstance(x, str) and x.startswith("["):
        return x.strip("[]").replace("'", "").split(",")[0].strip()
    return str(x)

df["speakers"] = df["speakers"].apply(flatten_speaker)

# Codificación de etiquetas.
le = LabelEncoder()
y = le.fit_transform(df["speakers"])
num_classes = len(le.classes_)

X = df["messages"].astype(str).values

# Codificación de etiquetas.
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Pesos de clase para tratar el desbalance.
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y),
    y=y
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)


# 2. DATASET
# (Estructura igual a la usada previamente).
class SpeakerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# 3. MODELO
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
).to(DEVICE)

# FREEZING MÁS AGRESIVO
# Se congelan: Loss embeddings y las primeras 4 capas del encoder.
# Esto reduce el número de parámetros entrenables y:
# - Mejora la estabilidad.
# - Reduce el riesgo de overfitting.
# - Acelera el entrenamiento.
# especialmente útil en datasets pequeños y homogéneos
for param in model.base_model.embeddings.parameters():
    param.requires_grad = False

for layer in model.base_model.encoder.layer[:4]:
    for param in layer.parameters():
        param.requires_grad = False

# 4. SETUP DE ENTRENAMIENTO
train_ds = SpeakerDataset(X_train, y_train, tokenizer)
val_ds = SpeakerDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LR)

# LABEL SMOOTHING
# Esto introduce incertidumbre controlada en las etiquetas.
# Ayuda a evitar sobreconfianza del modelo y mejorar generalización en multiclase.
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

# Scheduler con warm-up (igual que el modelo avanzado anterior).
total_steps = (len(train_loader) // ACCUM_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_steps),
    num_training_steps=total_steps
)

# Mixed precision
scaler = torch.cuda.amp.GradScaler()

# 5. BUCLE DE ENTRENAMIENTO
best_mcc = -1
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    train_loss = 0

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # Forward con mixed precision.
        with torch.amp.autocast("cuda"):
            outputs = model(input_ids, attention_mask=mask)
            loss = criterion(outputs.logits, labels)

            # Normalización de la loss para gradient accumulation.
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        # GRADIENT ACCUMULATION
        # Simula un batch mayor acumulando gradientes.
        # sin aumentar el consumo de memoria GPU.
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        train_loss += loss.item()

    # VALIDACIÓN (Igual que en los modelos anteriores).
    model.eval()
    y_true, y_pred = [], []
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids, attention_mask=mask)
            loss = criterion(outputs.logits, labels)

            val_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    val_loss /= len(val_loader)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    mcc = matthews_corrcoef(y_true, y_pred)

    print(
        f"Epoch {epoch+1} | Val Loss: {val_loss:.4f} | "
        f"MCC: {mcc:.4f} | F1: {f1:.4f} | Acc: {acc:.4f}"
    )

    # EARLY STOPPING BASADO EN MCC
    if mcc > best_mcc:
        best_mcc = mcc
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"Early stopping activado. Mejor MCC: {best_mcc:.4f}")
        break


Usando dispositivo: cuda


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Alba\AppData\Local\Temp\ipykernel_34600\1137246344.py:134: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1190/1190 [01:18<00:00, 15.11it/s]


Epoch 1 | Val Loss: 1.9889 | MCC: 0.0624 | F1: 0.1214 | Acc: 0.2337


Epoch 2/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1190/1190 [01:16<00:00, 15.47it/s]


Epoch 2 | Val Loss: 1.9055 | MCC: 0.1541 | F1: 0.2448 | Acc: 0.2665


...

Epoch 8 | Val Loss: 1.9466 | MCC: 0.2294 | F1: 0.3172 | Acc: 0.3405


Epoch 9/10: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1190/1190 [01:32<00:00, 12.91it/s]


Epoch 9 | Val Loss: 1.9802 | MCC: 0.2209 | F1: 0.3114 | Acc: 0.3333


Epoch 10/10: 100%|████████████████████████████████████████████████████████████████████████████████████████| 1190/1190 [01:31<00:00, 12.95it/s]


Epoch 10 | Val Loss: 1.9831 | MCC: 0.2313 | F1: 0.3208 | Acc: 0.3489


Los resultados muestran una evolución progresiva del modelo a lo largo de las epoch, alcanzando un F1-macro de 0.321, una accuracy cercana al 35% y un MCC máximo de 0.231. Aunque el entrenamiento es más estable y controlado que en las pruebas anteriores, el rendimiento final no supera al que hemos conseguido con la configuración anterior de RoBERTa-base. Esto quiere decir que el modelo empieza a alcanzar un límite de rendimiento para esta tarea, probablemente por la alta similitud semántica entre mensajes de distintos hablantes y al fuerte desbalanceo de los datos. En resumen, este últimom experimento confirma que hacer ajustes más agresivos en la optimización no siempre trae mejoras significativas.

### **Otros modelos**


### Data Augmentation con Llama 3

Tras observar que el desbalance de clases limita la capacidad de aprendizaje de los modelos de clasificación, vamos a analizar el uso de Llama 3 (arquitectura Decoder-only) mediante Ollama. Este modelo permite realizar Data Augmentation generando variantes sintéticas de mensajes engañosos.

Esta técnica permite equilibrar el dataset y usar modelos generativos capaces de replicar la lógica persuasiva y el léxico estratégico del juego Diplomacy.

In [3]:
import pandas as pd
import ollama
from tqdm import tqdm

# 1. Cargar el dataset que ya tiene el contexto (generado por tu script anterior)
df = pd.read_parquet("data/train_with_context.parquet")

# 2. Filtrar los mensajes que son mentiras (Clase minoritaria)
# Ajusta el nombre de la columna y el valor según tus etiquetas (ej: 'False' o '0')
lies_df = df[df["sender_labels"].astype(str).str.lower() == "false"]

def generate_variants(speaker, receiver, message, n=2):
    """
    Llama a Ollama para generar variantes estratégicas.
    """
    prompt = f"""
    Context: In a game of Diplomacy, {speaker} is lying to {receiver}.
    Original Message: "{message}"
    Task: Generate {n} different ways to say this same lie. 
    Keep the strategic tone. Respond ONLY with the messages, one per line.
    """
    try:
        response = ollama.generate(model='llama3', prompt=prompt)
        # Limpiamos la respuesta para obtener una lista de frases
        variants = [v.strip().strip('"') for v in response['response'].split('\n') if len(v) > 5]
        return variants[:n]
    except Exception as e:
        print(f"Error con Ollama: {e}")
        return []

# 3. Generar nuevas filas (Augmentation)
new_rows = []
print(f"Generando variantes para {len(lies_df)} mensajes de mentira...")

# Usamos un subconjunto para ahorrar tiempo
for _, row in tqdm(lies_df.iterrows(), total=len(lies_df)):
    variants = generate_variants(row['speakers'], row['receivers'], row['msg_for_context'])
    
    for v in variants:
        new_row = row.copy()
        new_row['msg_for_context'] = v
        # Re-generamos el text_context con el nuevo mensaje sintético
        new_row['text_context'] = f"{row['speakers']} -> {row['receivers']}: {v}"
        new_row['is_synthetic'] = True 
        new_rows.append(new_row)

# 4. Unir
df_augmented = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

Generando variantes para 522 mensajes de mentira...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 522/522 [21:18<00:00,  2.45s/it]


Dataset aumentado guardado. Filas totales: 12937


In [4]:
synthetic_examples = df_augmented[df_augmented['is_synthetic'] == True].head(10)

print("="*80)
print(f"{'ANÁLISIS DE MENSAJES SINTÉTICOS (LLAMA 3)':^80}")
print("="*80)

for i, (idx, row) in enumerate(synthetic_examples.iterrows()):
    # Buscamos el original (podemos usar el index o alguna referencia si la guardaste)
    # En este caso, simplemente mostramos el contenido del mensaje generado
    print(f"\nEJEMPLO {i+1}:")
    print(f"  País Emisor:   {row['speakers']}")
    print(f"  País Receptor: {row['receivers']}")
    print(f"  Mensaje Generado (Llama 3):")
    print(f"  > \"{row['msg_for_context']}\"")
    print("-" * 40)

# 2. Comparativa de balance de clases
print("\n" + "="*80)
print(f"{'BALANCE DE CLASES (SENDER_LABELS)':^80}")
print("="*80)
print(f"Dataset Original:  {df['sender_labels'].value_counts().to_dict()}")
print(f"Dataset Aumentado: {df_augmented['sender_labels'].value_counts().to_dict()}")
print("="*80)

                   ANÁLISIS DE MENSAJES SINTÉTICOS (LLAMA 3)                    

EJEMPLO 1:
  País Emisor:   italy
  País Receptor: germany
  Mensaje Generado (Llama 3):
  > "Here are two alternative ways to express Italy's lie:"
----------------------------------------

EJEMPLO 2:
  País Emisor:   italy
  País Receptor: germany
  Mensaje Generado (Llama 3):
  > "I've always advocated for a united front against Austria. We should focus on our shared interests rather than petty squabbles."
----------------------------------------

EJEMPLO 3:
  País Emisor:   italy
  País Receptor: germany
  Mensaje Generado (Llama 3):
  > "Here are two alternative ways to convey the same message:"
----------------------------------------

EJEMPLO 4:
  País Emisor:   italy
  País Receptor: germany
  Mensaje Generado (Llama 3):
  > "I've consistently communicated my intentions honestly and openly - is our alliance still intact?"
----------------------------------------

EJEMPLO 5:
  País Emisor:   italy


Podemos ver que se han generado nuevos ejemplos de mensajes engañosos, pero hay algunos que son partes de la conversación con el agente de ollama. Por ello haremos una limpieza para quedarnos solo con los mensajes del juego.

In [7]:
# Eliminamos filas que contienen frases típicas de la "charla" del modelo
chatter_patterns = ["Here are", "alternative ways", "convey the same", "message:"]
mask_chatter = df_augmented['msg_for_context'].str.contains('|'.join(chatter_patterns), case=False)

df_augmented = df_augmented[~mask_chatter].reset_index(drop=True)

# Actualizamos el text_context para las filas limpias
df_augmented['text_context'] = df_augmented['speakers'] + " -> " + df_augmented['receivers'] + ": " + df_augmented['msg_for_context']

print(f"Dataset tras limpieza de 'chatter': {len(df_augmented)} filas")
print(f"Nuevo balance 'False': {len(df_augmented[df_augmented['sender_labels'].astype(str).str.lower() == 'false'])}")

# Guardamos el dataset aumentado y limpio
output_path = "data/train_augmented_with_context.parquet"
df_augmented.to_parquet(output_path, index=False)

print(f"Archivo guardado en: {output_path}")
print(f"Filas totales: {len(df_augmented)}")

Dataset tras limpieza de 'chatter': 12574 filas
Nuevo balance 'False': 1230
Archivo guardado en: data/train_augmented_with_context.parquet
Filas totales: 12574


In [6]:
synthetic_examples = df_augmented[df_augmented['is_synthetic'] == True].head(10)

print("="*80)
print(f"{'ANÁLISIS DE MENSAJES SINTÉTICOS (LLAMA 3)':^80}")
print("="*80)

for i, (idx, row) in enumerate(synthetic_examples.iterrows()):
    # Buscamos el original (podemos usar el index o alguna referencia si la guardaste)
    # En este caso, simplemente mostramos el contenido del mensaje generado
    print(f"\nEJEMPLO {i+1}:")
    print(f"  País Emisor:   {row['speakers']}")
    print(f"  País Receptor: {row['receivers']}")
    print(f"  Mensaje Generado (Llama 3):")
    print(f"  > \"{row['msg_for_context']}\"")
    print("-" * 40)

# 2. Comparativa de balance de clases
print("\n" + "="*80)
print(f"{'BALANCE DE CLASES (SENDER_LABELS)':^80}")
print("="*80)
print(f"Dataset Original:  {df['sender_labels'].value_counts().to_dict()}")
print(f"Dataset Aumentado: {df_augmented['sender_labels'].value_counts().to_dict()}")
print("="*80)

                   ANÁLISIS DE MENSAJES SINTÉTICOS (LLAMA 3)                    

EJEMPLO 1:
  País Emisor:   italy
  País Receptor: germany
  Mensaje Generado (Llama 3):
  > "I've always advocated for a united front against Austria. We should focus on our shared interests rather than petty squabbles."
----------------------------------------

EJEMPLO 2:
  País Emisor:   italy
  País Receptor: germany
  Mensaje Generado (Llama 3):
  > "I've consistently communicated my intentions honestly and openly - is our alliance still intact?"
----------------------------------------

EJEMPLO 3:
  País Emisor:   italy
  País Receptor: germany
  Mensaje Generado (Llama 3):
  > "I've noticed our mutual interests are converging, and perhaps we've been overly accommodating towards each other."
----------------------------------------

EJEMPLO 4:
  País Emisor:   italy
  País Receptor: germany
  Mensaje Generado (Llama 3):
  > "I've come to realize that our goals align more closely than I initially tho

Vamos a probar el modelo de la tarea de clasificación con estos nuevos datos.

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# -------------------------------------
# Configuración Global
# -------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 10
PATIENCE = 3
LR = 1e-5 
MAX_LEN = 128
DATA_PATH = "data/train_augmented_with_context.parquet" 

models_to_compare = ["distilroberta-base"]

print(f"Usando dispositivo: {DEVICE}")

# -------------------------------------
# 1. Carga de Datos (Con Contexto y Augmentation)
# -------------------------------------
df = pd.read_parquet(DATA_PATH)

df['sender_labels'] = df['sender_labels'].astype(str).str.lower()
df = df[df["sender_labels"].isin(["true", "false"])]

y_raw = df["sender_labels"].map({"false": 0, "true": 1}).values

le = LabelEncoder()
df['target'] = le.fit_transform(y_raw)

X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    df["text_context"], df["target"], 
    stratify=df["target"], 
    test_size=0.2, 
    random_state=42
)

X_train, y_train = X_train_raw.values, y_train_raw.values
X_val, y_val = X_val_raw.values, y_val_raw.values

print(f"Datos preparados. Train: {len(X_train)} | Val: {len(X_val)}")
print(f"Ejemplo de entrada: {X_train[0]}")

# -------------------------------------
# 2. Pesos de Clase Suavizados
# -------------------------------------
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Pesos calculados: Clase 0 (Mentira): {class_weights[0]:.2f} | Clase 1 (Verdad): {class_weights[1]:.2f}")

# -------------------------------------
# 3. Dataset y Utilidades
# -------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = [str(t) for t in texts]
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer.encode_plus(
            self.texts[idx], 
            add_special_tokens=True, 
            max_length=self.max_len,
            padding='max_length', 
            truncation=True, 
            return_attention_mask=True, 
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def find_best_metrics(y_true, probs_class_0):
    best_thresh = 0.5
    best_mcc = -1
    best_acc = 0
    best_f1 = 0
    
    for thresh in np.arange(0.1, 0.9, 0.05):
        preds = np.where(probs_class_0 > thresh, 0, 1)
        mcc = matthews_corrcoef(y_true, preds)
        if mcc > best_mcc:
            best_mcc = mcc
            best_thresh = thresh
            best_acc = accuracy_score(y_true, preds)
            best_f1 = f1_score(y_true, preds, average='macro')
            
    return best_thresh, best_mcc, best_acc, best_f1

# -------------------------------------
# 4. Loop de Entrenamiento
# -------------------------------------
def train_model_weighted(model_name, X_train, y_train, X_val, y_val, class_weights_tensor):
    print(f"\n{'='*50}")
    print(f"FINE-TUNING: {model_name} (DATOS AUMENTADOS)")
    print(f"{'='*50}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(DEVICE)
    
    train_loader = DataLoader(TransformerDataset(X_train, y_train, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TransformerDataset(X_val, y_val, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)
    
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    best_mcc_val = -1
    final_metrics = {"acc": 0, "f1": 0}
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        model.train()
        for batch in train_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            targets = batch['labels'].to(DEVICE)
            
            outputs = model(ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, targets)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            
        model.eval()
        y_true, probs_0, total_val_loss = [], [], 0
        with torch.no_grad():
            for batch in val_loader:
                ids, mask, targets = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['labels'].to(DEVICE)
                outputs = model(ids, attention_mask=mask)
                total_val_loss += loss_fn(outputs.logits, targets).item()
                
                probs = F.softmax(outputs.logits, dim=1)
                probs_0.extend(probs[:, 0].cpu().numpy())
                y_true.extend(targets.cpu().numpy())
        
        y_true_np, probs_0_np = np.array(y_true), np.array(probs_0)
        thresh, mcc, acc, f1 = find_best_metrics(y_true_np, probs_0_np)
        
        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Epoch {epoch+1} | Loss: {avg_val_loss:.4f} | MCC: {mcc:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} (Th: {thresh:.2f})")
        
        if mcc > best_mcc_val:
            best_mcc_val = mcc
            final_metrics["acc"] = acc
            final_metrics["f1"] = f1
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print("Early Stopping: No hay mejora en MCC.")
            break

    print(f"\nRESULTADO FINAL - Mejor MCC: {best_mcc_val:.4f} | Acc: {final_metrics['acc']:.4f} | F1: {final_metrics['f1']:.4f}")

# -------------------------------------
# 5. Ejecución
# -------------------------------------
train_model_weighted(models_to_compare[0], X_train, y_train, X_val, y_val, weights_tensor)

Usando dispositivo: cuda
Datos preparados. Train: 10059 | Val: 2515
Ejemplo de entrada: germany -> italy: Dude idk I’m like freaked
Pesos calculados: Clase 0 (Mentira): 5.11 | Clase 1 (Verdad): 0.55

FINE-TUNING: distilroberta-base (DATOS AUMENTADOS)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 0.4326 | MCC: 0.6524 | Acc: 0.9451 | F1: 0.8192 (Th: 0.85)
Epoch 2 | Loss: 0.4732 | MCC: 0.6140 | Acc: 0.9340 | F1: 0.8067 (Th: 0.85)
Epoch 3 | Loss: 0.8036 | MCC: 0.7058 | Acc: 0.9535 | F1: 0.8408 (Th: 0.60)
Epoch 4 | Loss: 0.7253 | MCC: 0.6347 | Acc: 0.9368 | F1: 0.8172 (Th: 0.70)
Epoch 5 | Loss: 0.8237 | MCC: 0.6656 | Acc: 0.9459 | F1: 0.8292 (Th: 0.85)
Epoch 6 | Loss: 1.0118 | MCC: 0.4779 | Acc: 0.8680 | F1: 0.7218 (Th: 0.85)
Early Stopping: No hay mejora en MCC.

RESULTADO FINAL - Mejor MCC: 0.7058 | Acc: 0.9535 | F1: 0.8408


El MCC se ha incrementado drásticamente de 0.18 a 0.70, logrando una correlación fuerte y una detección de mentiras mucho más robusta.